In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [2]:
# ==========================================
# 1. THE CLIENT: LOCAL DATA & TRAINING
# ==========================================
class FLClient:
    def __init__(self, client_id, X, y):
        self.client_id = client_id
        self.X = X
        self.y = y
        self.weights = None
        self.intercept = None

    def train_locally(self, global_weights, global_intercept, lr=0.01, local_epochs=5):
        """
        Trains the local model starting from the global model's parameters.
        """
        # Step 1: Receive Global Model (Synchronization)
        self.weights = np.copy(global_weights)
        self.intercept = np.copy(global_intercept)
        
        # Step 2: Local Gradient Descent
        n_samples = self.X.shape[0]
        for _ in range(local_epochs):
            predictions = np.dot(self.X, self.weights) + self.intercept
            error = predictions - self.y
            
            # Calculate Gradients
            dw = (1 / n_samples) * np.dot(self.X.T, error)
            db = (1 / n_samples) * np.sum(error)
            
            # Update Local Parameters
            self.weights -= lr * dw
            self.intercept -= lr * db
            
        # Step 3: Send updated parameters back to server
        return self.weights, self.intercept, n_samples



In [3]:
# ==========================================
# 2. THE SERVER: AGGREGATION & COORDINATION
# ==========================================
class FLServer:
    def __init__(self, input_dim):
        # Initial global model
        self.global_weights = np.random.randn(input_dim) * 0.01
        self.global_intercept = 0.0

    def aggregate_and_broadcast(self, client_updates):
        """
        Applies FedAvg and prepares the model for the next round.
        """
        total_data_points = sum(update[2] for update in client_updates)
        
        new_weights = np.zeros_like(self.global_weights)
        new_intercept = 0.0
        
        for (w, b, n_k) in client_updates:
            # Calculate weight factor based on client's data contribution
            contribution_ratio = n_k / total_data_points
            new_weights += w * contribution_ratio
            new_intercept += b * contribution_ratio
            
        # Update Global Model
        self.global_weights = new_weights
        self.global_intercept = new_intercept



In [4]:
# ==========================================
# 3. THE SIMULATION PIPELINE
# ==========================================
def simulate_fl_system(num_clients=3, rounds=15):
    # Prepare House Price Dataset
    data = fetch_california_housing()
    X = StandardScaler().fit_transform(data.data)
    y = data.target
    
    # Split data among clients
    X_shards = np.array_split(X, num_clients)
    y_shards = np.array_split(y, num_clients)
    
    # Initialize Server and Clients
    server = FLServer(input_dim=X.shape[1])
    clients = [FLClient(i, X_shards[i], y_shards[i]) for i in range(num_clients)]
    
    print(f"--- Launching FL Simulation ---")
    
    for r in range(1, rounds + 1):
        collected_updates = []
        
        # Every client trains on the current global model
        for client in clients:
            update = client.train_locally(server.global_weights, server.global_intercept)
            collected_updates.append(update)
            
        # Server averages the updates
        server.aggregate_and_broadcast(collected_updates)
        
        # Periodic evaluation
        if r % 5 == 0 or r == 1:
            # Global Evaluation using full dataset (for simulation purposes)
            full_preds = np.dot(X, server.global_weights) + server.global_intercept
            mse = mean_squared_error(y, full_preds)
            print(f"Round {r:02d} | Global Model MSE: {mse:.4f}")

    print("\nFinal Global Intercept:", server.global_intercept)
    return server



In [5]:
if __name__ == "__main__":
    final_model = simulate_fl_system()

--- Launching FL Simulation ---
Round 01 | Global Model MSE: 5.1381
Round 05 | Global Model MSE: 3.6516
Round 10 | Global Model MSE: 2.4648
Round 15 | Global Model MSE: 1.7450

Final Global Intercept: 1.091801915726808
